# TreeMemory Colab Demo

This notebook runs the TreeMemory demo and benchmarks in Google Colab.

TreeMemory is an experimental hierarchical external memory layer for AI assistants. It stores facts in semantic branches, so retrieval can return cleaner context and local updates do not pollute nearby meanings.

## 1. Clone the Repository

This cell clones the public GitHub repository into Colab.

In [ ]:
REPO_URL = "https://github.com/g1g4b1t/tree-memory.git"

import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/tree-memory")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Cloned to", Path.cwd())

## 2. Install Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")

## 3. Run the Basic Demo

This shows the main idea: update one memory branch without corrupting a nearby branch.

In [ ]:
subprocess.run([sys.executable, "examples/demo.py"], check=True)

## 4. Run the Full Validation Suite

This checks syntax, unit tests, demo behavior, CLI smoke behavior, the original benchmark, and the scaled benchmark.

In [ ]:
subprocess.run([sys.executable, "scripts/validate.py"], check=True)

## 5. Inspect the Scaled Benchmark Results

In [ ]:
import json
import pandas as pd

with open("artifacts/results/scaled_memory_benchmark_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

overall = pd.DataFrame(results["overall"])
overall[[
    "memory",
    "top1_correct",
    "hit_at_k",
    "path_precision",
    "wrong_branch_hits",
    "context_contamination",
    "ai_context_risk",
]]

## 6. Try TreeMemory Directly

In [ ]:
from tree_memory_engine import build_demo_memory

memory = build_demo_memory()

for question in [
    "Who produces premium car tires?",
    "What are Michelin stars?",
    "For the reptile python, what does it shed?",
]:
    result = memory.answer(question)
    print(question, "->", result["answer"], "| path:", result["path"])

memory.update_fact(
    "artifacts/vehicles/car_tires",
    "car_tires.maker",
    "Bridgestone produces premium car tires in the updated memory.",
    "Bridgestone",
    "vehicle tires maker update",
)

print("\nAfter update:")
for question in [
    "Who produces premium car tires now?",
    "What are Michelin stars, not tires?",
]:
    result = memory.answer(question)
    print(question, "->", result["answer"], "| path:", result["path"])